# FEVER — Raw Data & Evaluation Analysis

## 1. Load dataset

In [1]:
import json
from pathlib import Path

import pandas as pd

pd.set_option('display.max_rows', None)
pd.set_option('display.max_columns', None)
pd.set_option('display.width', None)
pd.set_option('display.max_colwidth', None)

In [2]:
ROOT       = Path(".").resolve()
CACHE_FILE = ROOT / "data" / "fever" / "paper_dev.jsonl"

records = [json.loads(l) for l in CACHE_FILE.read_text().splitlines() if l.strip()]
print(f"Loaded {len(records):,} claims from {CACHE_FILE}")

Loaded 9,999 claims from /Users/bercaakbayir/Desktop/projects/unimi-dse-nlp-assignment/data/fever/paper_dev.jsonl


In [3]:
df = pd.DataFrame(records)
print(f"Shape: {df.shape}")
df.head(3)

Shape: (9999, 5)


,id,verifiable,label,claim,evidence
0,91198,NOT VERIFIABLE,NOT ENOUGH INFO,Colin Kaepernick became a starting quarterback during the 49ers 63rd season in the National Football League.,"[[[108548, None, None, None]]]"
1,194462,NOT VERIFIABLE,NOT ENOUGH INFO,Tilda Swinton is a vegan.,"[[[227768, None, None, None]]]"
2,137334,VERIFIABLE,SUPPORTS,Fox 2000 Pictures released the film Soul Food.,"[[[289914, 283015, Soul_Food_-LRB-film-RRB-, 0]], [[291259, 284217, Soul_Food_-LRB-film-RRB-, 0]], [[293412, 285960, Soul_Food_-LRB-film-RRB-, 0]], [[337212, 322620, Soul_Food_-LRB-film-RRB-, 0]], [[337214, 322622, Soul_Food_-LRB-film-RRB-, 0]]]"


In [13]:
df.head(10)

,id,verifiable,label,claim,evidence,claim_len,evidence_titles,n_evidence_titles
0,91198,NOT VERIFIABLE,NOT ENOUGH INFO,Colin Kaepernick became a starting quarterback during the 49ers 63rd season in the National Football League.,"[[[108548, None, None, None]]]",108,[],0
1,194462,NOT VERIFIABLE,NOT ENOUGH INFO,Tilda Swinton is a vegan.,"[[[227768, None, None, None]]]",25,[],0
2,137334,VERIFIABLE,SUPPORTS,Fox 2000 Pictures released the film Soul Food.,"[[[289914, 283015, Soul_Food_-LRB-film-RRB-, 0]], [[291259, 284217, Soul_Food_-LRB-film-RRB-, 0]], [[293412, 285960, Soul_Food_-LRB-film-RRB-, 0]], [[337212, 322620, Soul_Food_-LRB-film-RRB-, 0]], [[337214, 322622, Soul_Food_-LRB-film-RRB-, 0]]]",46,[Soul Food -LRB-film-RRB-],1
3,166626,NOT VERIFIABLE,NOT ENOUGH INFO,Anne Rice was born in New Jersey.,"[[[191656, None, None, None], [191657, None, None, None]]]",33,[],0
4,111897,VERIFIABLE,REFUTES,Telemundo is a English-language television network.,"[[[131371, 146144, Telemundo, 0]], [[131371, 146148, Telemundo, 1]], [[131371, 146150, Telemundo, 4], [131371, 146150, Hispanic_and_Latino_Americans, 0]], [[131371, 146151, Telemundo, 5]]]",51,"[Hispanic and Latino Americans, Telemundo]",2
5,89891,VERIFIABLE,REFUTES,Damon Albarn's debut album was released in 2011.,"[[[107201, 120581, Damon_Albarn, 17]]]",48,[Damon Albarn],1
6,181634,VERIFIABLE,SUPPORTS,There is a capital called Mogadishu.,"[[[210946, 218608, Mogadishu, 0]]]",36,[Mogadishu],1
7,219028,VERIFIABLE,REFUTES,Savages was exclusively a German film.,"[[[260471, 258880, Savages_-LRB-2012_film-RRB-, 3]], [[260473, 258882, Savages_-LRB-2012_film-RRB-, 3]]]",38,[Savages -LRB-2012 film-RRB-],1
8,194372,NOT VERIFIABLE,NOT ENOUGH INFO,Happiness in Slavery is a gospel song by Nine Inch Nails.,"[[[227658, None, None, None]]]",57,[],0
9,108281,VERIFIABLE,REFUTES,Andrew Kevin Walker is only Chinese.,"[[[127089, 141573, Andrew_Kevin_Walker, 0]]]",36,[Andrew Kevin Walker],1


## 2. Schema

In [4]:
df.dtypes

id             int64
verifiable       str
label            str
claim            str
evidence      object
dtype: object

In [5]:
# One full example — shows nested evidence structure
ex = df.iloc[0]
print("id         :", ex["id"])
print("label      :", ex["label"])
print("claim      :", ex["claim"])
print()
print("evidence   :")
for ann_set in ex["evidence"]:
    for piece in ann_set:
        print("  ", piece)  # [ann_id, ev_id, wiki_title_or_null, sent_id]

id         : 91198
label      : NOT ENOUGH INFO
claim      : Colin Kaepernick became a starting quarterback during the 49ers 63rd season in the National Football League.

evidence   :
   [108548, None, None, None]


## 3. Basic statistics

In [6]:
print("=== Label distribution ===")
print(df["label"].value_counts())
print()
print("=== Label distribution (%) ===")
print(df["label"].value_counts(normalize=True).mul(100).round(1))

=== Label distribution ===
label
NOT ENOUGH INFO    3333
SUPPORTS           3333
REFUTES            3333
Name: count, dtype: int64

=== Label distribution (%) ===
label
NOT ENOUGH INFO    33.3
SUPPORTS           33.3
REFUTES            33.3
Name: proportion, dtype: float64


In [7]:
df["claim_len"] = df["claim"].str.len()

print("=== Claim length (chars) ===")
print(df["claim_len"].describe().round(1))

=== Claim length (chars) ===
count    9999.0
mean       47.7
std        17.9
min        13.0
25%        35.0
50%        45.0
75%        57.0
max       219.0
Name: claim_len, dtype: float64


In [8]:
print("=== Claim length by label ===")
print(df.groupby("label")["claim_len"].describe().round(1))

=== Claim length by label ===
                  count  mean   std   min   25%   50%   75%    max
label                                                             
NOT ENOUGH INFO  3333.0  48.6  18.2  15.0  35.0  46.0  58.0  219.0
REFUTES          3333.0  47.2  16.6  14.0  35.0  45.0  56.0  120.0
SUPPORTS         3333.0  47.5  18.7  13.0  34.0  44.0  56.0  200.0


## 4. Evidence analysis

In [9]:
def extract_evidence_titles(evidence):
    """Return list of unique Wikipedia titles from FEVER evidence field."""
    titles = set()
    for ann_set in evidence:
        for piece in ann_set:
            if len(piece) >= 3 and piece[2]:
                titles.add(piece[2].replace("_", " "))
    return sorted(titles)

df["evidence_titles"] = df["evidence"].apply(extract_evidence_titles)
df["n_evidence_titles"] = df["evidence_titles"].apply(len)

print("=== Unique evidence titles per claim ===")
print(df["n_evidence_titles"].value_counts().sort_index())

=== Unique evidence titles per claim ===
n_evidence_titles
0     3333
1     5772
2      693
3      100
4       38
5       21
6       14
7        4
8       12
9        1
11       3
12       2
14       1
15       4
16       1
Name: count, dtype: int64


In [10]:
print("=== Evidence titles count by label ===")
print(df.groupby("label")["n_evidence_titles"].describe().round(2))

=== Evidence titles count by label ===
                  count  mean   std  min  25%  50%  75%   max
label                                                        
NOT ENOUGH INFO  3333.0  0.00  0.00  0.0  0.0  0.0  0.0   0.0
REFUTES          3333.0  1.19  0.79  1.0  1.0  1.0  1.0  15.0
SUPPORTS         3333.0  1.23  0.83  1.0  1.0  1.0  1.0  16.0


In [11]:
# Unique Wikipedia titles across the full dev set
all_titles = set(t for titles in df["evidence_titles"] for t in titles)
print(f"Total unique evidence Wikipedia titles: {len(all_titles):,}")

Total unique evidence Wikipedia titles: 1,460


In [12]:
# Most frequently cited Wikipedia pages
from collections import Counter

title_counts = Counter(t for titles in df["evidence_titles"] for t in titles)
print("=== Top 20 most cited Wikipedia pages ===")
for title, count in title_counts.most_common(20):
    print(f"  {count:4d}  {title}")

=== Top 20 most cited Wikipedia pages ===
    25  Uranium-235
    24  CHiPs -LRB-film-RRB-
    23  Reign Over Me
    23  Camden, New Jersey
    23  Carlos Santana
    22  Trollhunters
    22  T2 Trainspotting
    21  Harold Macmillan
    21  Colombiana
    21  Finding Dory
    21  French Indochina
    21  Miranda Otto
    21  Wish Upon
    21  The Endless River
    21  Billboard Dad
    21  Happiness in Slavery
    20  Edgar Wright
    20  Despicable Me 2
    20  The Mighty Ducks
    20  Trevor Griffiths


## 5. Sample claims by label

In [ ]:
print("=== SUPPORTS (sample) ===")
sample = df[df["label"] == "SUPPORTS"][["claim", "evidence_titles"]].sample(5, random_state=42)
for _, r in sample.iterrows():
    print(f"  Claim    : {r['claim']}")
    print(f"  Evidence : {r['evidence_titles']}")
    print()

In [ ]:
print("=== REFUTES (sample) ===")
sample = df[df["label"] == "REFUTES"][["claim", "evidence_titles"]].sample(5, random_state=42)
for _, r in sample.iterrows():
    print(f"  Claim    : {r['claim']}")
    print(f"  Evidence : {r['evidence_titles']}")
    print()

In [ ]:
print("=== NOT ENOUGH INFO (sample) ===")
sample = df[df["label"] == "NOT ENOUGH INFO"][["claim"]].sample(5, random_state=42)
for _, r in sample.iterrows():
    print(f"  Claim: {r['claim']}")
    print()

## 6. Wiki corpus coverage

In [ ]:
WIKI_CACHE = ROOT / "data" / "fever" / "wiki_pages.jsonl"

if WIKI_CACHE.exists():
    wiki_pages = [json.loads(l) for l in WIKI_CACHE.read_text().splitlines() if l.strip()]
    wiki_df = pd.DataFrame(wiki_pages)
    wiki_df["text_len"] = wiki_df["text"].str.len()

    fetched_titles = set(wiki_df["title"])
    coverage = len(fetched_titles & all_titles) / len(all_titles) * 100

    print(f"Pages in corpus     : {len(wiki_df):,}")
    print(f"Unique ev. titles   : {len(all_titles):,}")
    print(f"Corpus coverage     : {coverage:.1f}%")
    print()
    print("=== Page text length (chars) ===")
    print(wiki_df["text_len"].describe().round(1))
else:
    print("wiki_pages.jsonl not found — run build_fever_db.py first")

## 7. Evaluation analysis

Run `python src/eval/evaluate_fever.py --limit 100 --output results/fever_smoke.jsonl` first.

In [ ]:
import glob

# Auto-detect the most recent fever evaluation JSONL
result_files = sorted(glob.glob(str(ROOT / "results" / "fever_eval*.jsonl")))
if not result_files:
    result_files = sorted(glob.glob(str(ROOT / "results" / "fever*.jsonl")))

if not result_files:
    print("No fever evaluation results found.")
    print("Run: python src/eval/evaluate_fever.py --limit 100 --output results/fever_smoke.jsonl")
else:
    latest = result_files[-1]
    print(f"Loading: {latest}")
    eval_records = [json.loads(l) for l in Path(latest).read_text().splitlines() if l.strip()]
    eval_df = pd.DataFrame(eval_records)
    print(f"Evaluated claims: {len(eval_df):,}")
    eval_df.head(5)

In [ ]:
if 'eval_df' in dir() and len(eval_df) > 0:
    ok = eval_df[eval_df["error"].isna()]

    print("=== Overall metrics ===")
    print(f"  Label Accuracy  : {ok['label_acc'].mean()*100:.1f}%")
    print(f"  FEVER Score     : {ok['fever_sc'].mean()*100:.1f}%")
    print(f"  Evidence P      : {ok['ev_precision'].mean()*100:.1f}%")
    print(f"  Evidence R      : {ok['ev_recall'].mean()*100:.1f}%")
    print(f"  Evidence F1     : {ok['ev_f1'].mean()*100:.1f}%")

In [ ]:
if 'eval_df' in dir() and len(eval_df) > 0:
    ok = eval_df[eval_df["error"].isna()]

    print("=== Metrics by gold label ===")
    for label in ["SUPPORTS", "REFUTES", "NOT ENOUGH INFO"]:
        sub = ok[ok["gold_label"] == label]
        if len(sub) == 0:
            continue
        print(f"\n  {label} (n={len(sub)})")
        print(f"    Label Accuracy : {sub['label_acc'].mean()*100:.1f}%")
        print(f"    FEVER Score    : {sub['fever_sc'].mean()*100:.1f}%")
        print(f"    Evidence F1    : {sub['ev_f1'].mean()*100:.1f}%")

In [ ]:
if 'eval_df' in dir() and len(eval_df) > 0:
    ok = eval_df[eval_df["error"].isna()].copy()

    # Normalise predicted labels for confusion matrix
    def _norm(s):
        s = str(s).upper().strip()
        if "NOT ENOUGH" in s or "NEI" in s: return "NEI"
        if "REFUT" in s: return "REFUTES"
        if "SUPPORT" in s: return "SUPPORTS"
        return "OTHER"

    ok["pred_norm"] = ok["pred_label"].apply(_norm)
    ok["gold_norm"] = ok["gold_label"].apply(lambda x: "NEI" if x == "NOT ENOUGH INFO" else x)

    confusion = pd.crosstab(
        ok["gold_norm"], ok["pred_norm"],
        rownames=["Gold"], colnames=["Predicted"]
    )
    print("=== Confusion matrix ===")
    print(confusion)

In [ ]:
if 'eval_df' in dir() and len(eval_df) > 0:
    ok = eval_df[eval_df["error"].isna()]

    print("=== Errors / skipped ===")
    errors = eval_df[eval_df["error"].notna()]
    print(f"  {len(errors)} error(s) out of {len(eval_df)} total")
    if len(errors):
        print(errors[["id", "claim", "error"]].head(5))

In [ ]:
if 'eval_df' in dir() and len(eval_df) > 0:
    ok = eval_df[eval_df["error"].isna()]

    # Show wrong predictions for SUPPORTS claims
    wrong = ok[(ok["gold_label"] == "SUPPORTS") & (ok["label_acc"] == 0)]
    print(f"=== Wrong SUPPORTS predictions (sample of up to 5) ===")
    for _, r in wrong.head(5).iterrows():
        print(f"  Claim     : {r['claim']}")
        print(f"  Predicted : {r['pred_label']}")
        print(f"  Gold      : {r['gold_label']}")
        print()

In [ ]:
# Load CSV export if available
csv_files = sorted(glob.glob(str(ROOT / "results" / "fever_eval_*.csv")))
if csv_files:
    csv_df = pd.read_csv(csv_files[-1])
    print(f"CSV: {csv_files[-1]}")
    csv_df.head(10)
else:
    print("No CSV found — will be created when evaluate_fever.py finishes.")